# 東坡 LoRA Adapter 驗收 Notebook

**筆記**: 用於2026.06.08, 看看目前LoRA Train結果

**用途**：序列式對比「純東坡」vs「東坡 + 同學的 adapter」，判斷這份 LoRA 值不值得用。

**設計**：每個動作獨立 cell，手動控制讀取/卸載。8GB 顯存一次只放一個模型。

**使用順序建議**：
1. 跑 `[環境]` 兩格（安裝 + 設定）
2. 跑 `[路徑]` 填好你的路徑
3. 跑 `[問題集]` 設定要問的題目
4. `[A] 讀取純東坡` → `[A] 問答` → 收集答案 → `[卸載]`
5. `[B] 讀取東坡+adapter` → `[B] 問答` → 收集答案 → `[卸載]`
6. `[對比] 並排輸出`

本地 Jupyter/VSCode 或 Colab 都可直接用同一份。

---
## [環境] 安裝套件
本地若已裝好可跳過。Colab 第一次要跑。

In [ ]:
# Colab / 全新環境執行。本地已備妥可跳過此格。
%pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
# 本地(3070)環境 降板:
%pip install "transformers==4.46.3" "tokenizers==0.20.3" "peft==0.13.2"

In [ ]:
# 本地(3070)環境 除錯:
import transformers, tokenizers, peft, torch
print("transformers", transformers.__version__)
print("tokenizers  ", tokenizers.__version__)
print("peft        ", peft.__version__)
print("torch       ", torch.__version__)
import bitsandbytes; print("bitsandbytes", bitsandbytes.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

transformers 4.46.3
tokenizers   0.20.3
peft         0.13.2
torch        2.5.1+cu121
bitsandbytes 0.49.2


---
## [環境:共通] 匯入與共用設定

 注意: Colab可能需要特別使用
 T4 是比較舊的架構（Turing），<br>
 硬體層面並不支援 bfloat16。<br>
 如果在 T4 上強制使用 bfloat16，PyTorch <br>
 會用軟體模擬（ fallback to FP32），<br>
 這會導致你的推論速度變得極慢，甚至引發錯誤。


In [2]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"裝置：{DEVICE}")
if DEVICE == "cuda":
    print(f"GPU：{torch.cuda.get_device_name(0)}")
    print(f"顯存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# 4-bit 量化設定（8GB 顯存載入 8B 模型必須）3070使用 bfloat16
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 4-bit 量化設定（8GB 顯存載入 8B 模型必須）T4使用 float16
# BNB_CONFIG = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16, # <-- 配合 T4 改成 float16
#     bnb_4bit_use_double_quant=True,
# )

# 用來追蹤目前載入的是哪個模型，避免重複載入爆顯存
CURRENT = {"model": None, "tokenizer": None, "which": None}

print("設定完成")

裝置：cuda
GPU：NVIDIA GeForce RTX 3070
顯存：8.0 GB
設定完成


## [環境:共通] 或者, 可以直接嘗試使用GPT提供的自動切換

In [3]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"裝置：{DEVICE}")

if DEVICE == "cuda":
    print(f"GPU：{torch.cuda.get_device_name(0)}")
    print(f"顯存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

    major, minor = torch.cuda.get_device_capability()

    if major >= 8:
        COMPUTE_DTYPE = torch.bfloat16
        print("使用 bfloat16")
    else:
        COMPUTE_DTYPE = torch.float16
        print("使用 float16")
else:
    COMPUTE_DTYPE = torch.float16

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

CURRENT = {
    "model": None,
    "tokenizer": None,
    "which": None
}

print("設定完成")

裝置：cuda
GPU：NVIDIA GeForce RTX 3070
顯存：8.0 GB
使用 bfloat16
設定完成


---
## [路徑] 填入你的路徑

- `BASE_MODEL`：東坡底模。可直接用 HF 名稱 `QingYuYunTu/DongPo`（會自動下載），或填你本地/Drive 的資料夾路徑。
- `ADAPTER_PATH`：同學那包 adapter 的資料夾（裡面要有 `adapter_config.json` + `adapter_model.safetensors`）。

跑這格會驗證路徑/檔案存在，正確才印出確認字串。

In [ ]:
import os
import sys

# 1. 自動偵測運行環境
IN_COLAB = 'google.colab' in sys.modules

# 2. 依據環境分配路徑與掛載硬碟
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # === Colab 路徑配置 ===
    BASE_MODEL   = "QingYuYunTu/DongPo"  # 建議直接填寫 HuggingFace Repo，讓 Colab 自動下載暫存
    ADAPTER_PATH = "/content/drive/MyDrive/FJU_dongpoProject/adapter" # 替換成你雲端硬碟的實際路徑
else:
    # === 本地端 (RTX 3070) 路徑配置 ===
    # 本地端直接略過 drive.mount
    BASE_MODEL   = "QingYuYunTu/DongPo"  # 同樣可填 HF Repo，或填寫本地存放基模的絕對路徑
    ADAPTER_PATH = r"E:\FJU_dongpoProject\RAG\500plus150RAG_byGOUNG_0531\500plus150RAGV0531"

# 3. 執行驗證邏輯 (保留你原本的寫法即可)
assert BASE_MODEL.strip(), "BASE_MODEL 還沒填"
assert ADAPTER_PATH.strip(), "ADAPTER_PATH 還沒填"

assert os.path.isdir(ADAPTER_PATH), f"找不到 adapter 資料夾：{ADAPTER_PATH}"
assert os.path.isfile(os.path.join(ADAPTER_PATH, "adapter_config.json")), "adapter 資料夾缺 adapter_config.json"
_has_weight = any(
    os.path.isfile(os.path.join(ADAPTER_PATH, f))
    for f in ("adapter_model.safetensors", "adapter_model.bin")
)
assert _has_weight, "adapter 資料夾缺 adapter_model.safetensors / .bin"

if os.path.sep in BASE_MODEL or BASE_MODEL.startswith("."):
    assert os.path.exists(BASE_MODEL), f"找不到底模路徑：{BASE_MODEL}"

print(f"環境判斷: {'Colab' if IN_COLAB else '本地端'}")
print("路徑正確讀取完畢")

---
## [模板] 東坡專用對話格式

東坡底模用的是非標準的 `System: / Human: / Assistant:` 格式（**不是** Qwen 原生 ChatML）。
這裡手動實作，確保推論格式跟訓練/底模一致。

In [ ]:
IM_END = "<|im_end|>"

def build_prompt(user_msg, system_msg=None):
    """複刻東坡 chat_template.jinja 的 Human/Assistant 格式。"""
    s = ""
    if system_msg:
        s += f"System: {system_msg}{IM_END}\n"
    s += f"Human: {user_msg}{IM_END}\nAssistant:"
    return s

@torch.inference_mode()
def ask(model, tokenizer, user_msg, system_msg=None, max_new_tokens=256, temperature=0.7):
    prompt = build_prompt(user_msg, system_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.manual_seed(42) 
    # 固定亂數種子（Seed）以控制變因
    # 實驗目的是「對比純底模與加上 Adapter 的差異」。
    
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=0.8,
        eos_token_id=tokenizer.convert_tokens_to_ids(IM_END),
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(gen, skip_special_tokens=True)
    return text.strip()

print("模板與問答函式就緒")

---
## [問題集] 設定要對比的問題

建議混入：純風格題（測人格濃度）、史實題（測知識）、現代物題（測「不知今事」設定有沒有被 adapter 破壞）。
答案會分別存進 `answers_A` / `answers_B`。

In [ ]:
QUESTIONS = [
    "先生為何自號東坡居士？",
    "可曾後悔貶謫黃州？",
    "念奴嬌·赤壁懷古作於何時？",
    "智慧型手機是什麼？",          # 測現代無知設定
    "先生對人生無常有何看法？",
]

SYSTEM_MSG = None  # 東坡角色已固化進權重，通常不需 system prompt；要測可自行填

answers_A = {}  # 純東坡
answers_B = {}  # 東坡 + adapter
print(f"已設定 {len(QUESTIONS)} 題")

---
## [卸載] 清空目前模型

**換模型前必跑這格。** 把顯存吐乾淨，否則 8GB 載完 A 直接載 B 會 OOM。
可重複跑，安全。

In [ ]:
def unload():
    if CURRENT["model"] is not None:
        was = CURRENT["which"]
        del CURRENT["model"]
        CURRENT["model"] = None
        # tokenizer 不佔顯存，留著無妨，但一併清乾淨
        CURRENT["tokenizer"] = None
        CURRENT["which"] = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print(f"已卸載：{was}")
    else:
        print("目前沒有載入任何模型")
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1024**3
        print(f"目前顯存佔用：{used:.2f} GB")

unload()

---
## [A] 讀取純東坡
載入前若已有模型，請先跑 `[卸載]`。

In [ ]:
assert CURRENT["model"] is None, "已有模型載入中，請先跑 [卸載] 那格"

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
)
mdl.eval()
CURRENT.update(model=mdl, tokenizer=tok, which="A:純東坡")
print("已載入 A：純東坡")
if torch.cuda.is_available():
    print(f"顯存佔用：{torch.cuda.memory_allocated()/1024**3:.2f} GB")

## [A] 問答（純東坡）
答案存進 `answers_A`。

In [ ]:
assert CURRENT["which"] == "A:純東坡", "目前載入的不是 A，請確認"
for q in QUESTIONS:
    a = ask(CURRENT["model"], CURRENT["tokenizer"], q, system_msg=SYSTEM_MSG)
    answers_A[q] = a
    print(f"問：{q}\n答：{a}\n{'-'*60}")

---
## [B] 讀取東坡 + adapter
**載入前務必先跑 `[卸載]` 把 A 清掉。**

In [ ]:
assert CURRENT["model"] is None, "已有模型載入中，請先跑 [卸載] 那格"

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
)
mdl = PeftModel.from_pretrained(base, ADAPTER_PATH)
mdl.eval()
CURRENT.update(model=mdl, tokenizer=tok, which="B:東坡+adapter")
print("已載入 B：東坡 + adapter")
if torch.cuda.is_available():
    print(f"顯存佔用：{torch.cuda.memory_allocated()/1024**3:.2f} GB")

## [B] 問答（東坡 + adapter）
答案存進 `answers_B`。

In [ ]:
assert CURRENT["which"] == "B:東坡+adapter", "目前載入的不是 B，請確認"
for q in QUESTIONS:
    a = ask(CURRENT["model"], CURRENT["tokenizer"], q, system_msg=SYSTEM_MSG)
    answers_B[q] = a
    print(f"問：{q}\n答：{a}\n{'-'*60}")

---
## [對比] 並排輸出
兩邊都跑完後執行。可在卸載 B 之後跑（純讀字典，不需模型在顯存裡）。

In [ ]:
for q in QUESTIONS:
    print("="*70)
    print(f"【問題】{q}\n")
    print(f"〔A 純東坡〕\n{answers_A.get(q, '（尚未作答）')}\n")
    print(f"〔B 東坡+adapter〕\n{answers_B.get(q, '（尚未作答）')}\n")
print("="*70)
print("對比結束。重點看：B 的東坡風格有沒有更濃、史實有沒有更準、")
print("以及現代物題 B 有沒有破功（adapter 若混入現代知識會破壞『不知今事』設定）。")

---
## [自由問答] 臨時問單題
對目前載入的模型（A 或 B 皆可）隨手問一題，不存進字典。

In [ ]:
assert CURRENT["model"] is not None, "目前沒有載入模型"
my_q = "先生最愛何種食物？"   # 改成你想問的
print(f"（目前模型：{CURRENT['which']}）")
print(ask(CURRENT["model"], CURRENT["tokenizer"], my_q, system_msg=SYSTEM_MSG))